In [1]:
import scanpy as sc
import scanpy as sc
import numpy as np
import anndata as ad

In [3]:
adata = sc.read_h5ad("multiperturb_seq.h5ad")
print(adata.shape)
print(adata.obs.columns)

(121651, 46597)
Index(['guide_id', 'guide_gene'], dtype='object')


In [4]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor='seurat')
sc.tl.pca(adata, n_comps=30, use_highly_variable=True)

In [7]:
# If pseudotime is not already a column, compute it from PCA -> first PC
if 'pseudotime' not in adata.obs:
    print("Computing pseudotime from PCA first component...")
    if 'X_pca' not in adata.obsm:
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)
        sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor='seurat')
        sc.tl.pca(adata, n_comps=30, use_highly_variable=True)
    pseudotime = adata.obsm['X_pca'][:, 0]
    pseudotime = (pseudotime - pseudotime.min()) / (pseudotime.max() - pseudotime.min())
    adata.obs['pseudotime'] = pseudotime
print("Pseudotime range:", pseudotime.min(), pseudotime.max())


Pseudotime range: 0.0 1.0


In [11]:
# For final entire dataset
#sc.pp.neighbors(adata)
#sc.tl.diffmap(adata)
#pseudotime = adata.obsm['X_diffmap'][:, 0]
#pseudotime = (pseudotime - pseudotime.min()) / (pseudotime.max() - pseudotime.min())
#adata.obs['pseudotime'] = pseudotime

In [8]:
print(adata.obs['guide_gene'].unique())
print(adata.obs['guide_gene'].value_counts())

['BAZ1B', 'ACTL6A', 'non-targeting (mouse)', 'PHF10', 'GATAD2A', ..., 'SMARCD2', 'SMARCC2', 'SUPT16H', 'KMT2C', 'SETD1A']
Length: 101
Categories (101, object): ['ACTL6A', 'ACTL6B', 'ACTR5', 'ARID1B', ..., 'ZBTB18', 'ZNHIT1', 'non-targeting (human)', 'non-targeting (mouse)']
non-targeting (mouse)    19832
BAZ1B                     9154
non-targeting (human)     5293
PRDM16                    3815
SMARCE1                   3394
                         ...  
CHD3                         1
SETD1A                       1
BRD2                         1
YY1                          1
KMT2C                        1
Name: guide_gene, Length: 101, dtype: int64


In [9]:
control_label = 'non-targeting (mouse)'
perturb_label = 'SMARCE1'

adata_ctl = adata[adata.obs['guide_gene'] == control_label].copy()
adata_pert = adata[adata.obs['guide_gene'] == perturb_label].copy()
print(f"Control cells: {adata_ctl.n_obs}, Perturbed cells: {adata_pert.n_obs}")

Control cells: 19832, Perturbed cells: 3394


In [10]:
adata_ctl.write("control_multiperturb.h5ad")

In [20]:
adata = sc.read_h5ad("multiperturb_seq.h5ad")

In [11]:
# Add a copy of the ATAC matrix to the expected key
adata.obsm['ATAC_gene'] = adata.obsm['ATAC']

In [12]:
# Subset control and perturbed cells
control_label = 'non-targeting (mouse)'
perturb_label = 'SMARCE1'

adata_ctl = adata[adata.obs['guide_gene'] == control_label].copy()
adata_pert = adata[adata.obs['guide_gene'] == perturb_label].copy()

print(f"Control cells: {adata_ctl.n_obs}, Perturbed cells: {adata_pert.n_obs}")

Control cells: 19832, Perturbed cells: 3394


In [15]:
# Subsample control cells (optional but recommended)
np.random.seed(42)
n_ctl_subsample = 5000
idx_ctl = np.random.choice(adata_ctl.n_obs, n_ctl_subsample, replace=False)
adata_ctl_small = adata_ctl[idx_ctl].copy()
adata_ctl_small.write("control_multiperturb_small.h5ad")

In [1]:
import scanpy as sc
adata = sc.read_h5ad("multiperturb_seq.h5ad")
# Rename the column
adata.obs.rename(columns={'guide_gene': 'guide_target'}, inplace=True)
# Also ensure 'ATAC_gene' exists
if 'ATAC_gene' not in adata.obsm:
    adata.obsm['ATAC_gene'] = adata.obsm['ATAC']
# Save
adata.write("multiperturb_seq_ready.h5ad")

In [1]:
import scanpy as sc
import numpy as np

# Load the fixed AnnData
adata = sc.read_h5ad("multiperturb_seq_ready.h5ad")

# Subsample to 10,000 cells (or adjust)
np.random.seed(42)
n_subsample = 10000
if adata.n_obs > n_subsample:
    idx = np.random.choice(adata.n_obs, n_subsample, replace=False)
    adata_sub = adata[idx].copy()
else:
    adata_sub = adata

# Keep the same column names (guide_target already exists)
# Save the subsampled object
adata_sub.write("multiperturb_seq_subsampled.h5ad")
print("Saved subsampled AnnData")

Saved subsampled AnnData


In [1]:
import scanpy as sc
import numpy as np

# Load the original file (which has pseudotime)
adata_orig = sc.read_h5ad("multiperturb_seq_ready.h5ad")

# Load the subsampled file (which lacks pseudotime)
adata_sub = sc.read_h5ad("multiperturb_seq_subsampled.h5ad")

# Transfer pseudotime from original to subsampled by matching cell barcodes
# Since subsampled is a subset of original, the barcodes are identical.
# We can simply copy the column from the original for the same indices.
# However, it's easier to recreate the subsample with the original file.
np.random.seed(42)
n_subsample = 10000
if adata_orig.n_obs > n_subsample:
    idx = np.random.choice(adata_orig.n_obs, n_subsample, replace=False)
    adata_sub_new = adata_orig[idx].copy()
else:
    adata_sub_new = adata_orig.copy()

# Verify pseudotime is present
print("Pseudotime in subsampled:", 'pseudotime' in adata_sub_new.obs)

# Save
adata_sub_new.write("multiperturb_seq_subsampled_with_pt.h5ad")

Pseudotime in subsampled: False


In [2]:
import scanpy as sc
adata_orig = sc.read_h5ad("multiperturb_seq_ready.h5ad")
print("Has pseudotime?", 'pseudotime' in adata_orig.obs)

Has pseudotime? False


In [3]:
import scanpy as sc
import numpy as np
import pandas as pd

# Load the original file
adata = sc.read_h5ad("multiperturb_seq.h5ad")
print("Original shape:", adata.shape)
print("Columns in obs:", adata.obs.columns.tolist()[:10])

Original shape: (121651, 46597)
Columns in obs: ['guide_id', 'guide_gene']


In [4]:
if 'pseudotime' not in adata.obs:
    print("Computing pseudotime...")
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor='seurat')
    sc.tl.pca(adata, n_comps=30, use_highly_variable=True)
    pseudotime = adata.obsm['X_pca'][:, 0]
    pseudotime = (pseudotime - pseudotime.min()) / (pseudotime.max() - pseudotime.min())
    adata.obs['pseudotime'] = pseudotime
    # Save the updated full dataset (optional)
    adata.write("multiperturb_seq_with_pt.h5ad")
else:
    print("Pseudotime already present.")

# Verify
print("Has pseudotime?", 'pseudotime' in adata.obs)

Computing pseudotime...
Has pseudotime? True


In [5]:
np.random.seed(42)
n_subsample = 10000
if adata.n_obs > n_subsample:
    idx = np.random.choice(adata.n_obs, n_subsample, replace=False)
    adata_sub = adata[idx].copy()
else:
    adata_sub = adata.copy()

# Ensure the necessary keys are present
if 'guide_target' not in adata_sub.obs and 'guide_gene' in adata_sub.obs:
    adata_sub.obs.rename(columns={'guide_gene': 'guide_target'}, inplace=True)
if 'ATAC_gene' not in adata_sub.obsm and 'ATAC' in adata_sub.obsm:
    adata_sub.obsm['ATAC_gene'] = adata_sub.obsm['ATAC']

# Save
adata_sub.write("multiperturb_seq_subsampled_with_pt.h5ad")
print("Saved subsampled file.")

Saved subsampled file.


In [1]:
import scanpy as sc
adata = sc.read_h5ad("multiperturb_seq_subsampled_with_pt.h5ad")
print("First 10 gene names:", adata.var_names[:10])
print("Search for Cdkn1a-like:", [g for g in adata.var_names if 'Cdkn1a' in g or 'cdkn1a' in g])
print("Search for Myc:", [g for g in adata.var_names if 'Myc' in g])

First 10 gene names: Index(['0', '1', '2', '3', '4', '5', '6', '7', '8', '9'], dtype='object')
Search for Cdkn1a-like: []
Search for Myc: []


In [2]:
print(adata.var.columns)
print(adata.var.head())

Index(['gene', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'], dtype='object')
       gene  highly_variable     means  dispersions  dispersions_norm
0    TSPAN6             True  0.091549     9.048012          1.567667
1      TNMD            False  0.000017     0.406021         -2.014950
2      DPM1            False  0.299516     8.739242          1.439664
3     SCYL3            False  0.248685     8.639825          1.398449
4  C1orf112            False  0.190324     8.239818          1.232623


In [3]:
# Example if column is 'gene'
gene_names = adata.var['gene'].values
print(gene_names[:20])

['TSPAN6', 'TNMD', 'DPM1', 'SCYL3', 'C1orf112', ..., 'ANKIB1', 'CYP51A1', 'KRIT1', 'RAD52', 'BAD']
Length: 20
Categories (41761, object): ['', '0610006L08Rik', '0610009B22Rik', '0610009E02Rik', ..., 'mt-Nd4', 'mt-Nd4l', 'mt-Nd5', 'mt-Nd6']


In [4]:
import scanpy as sc
adata = sc.read_h5ad("multiperturb_seq_subsampled_with_pt.h5ad")
genes = adata.var['gene'].values
targets = ['Cdkn1a', 'Myc', 'E2f1', 'Ccnd1', 'Cdk2']
found = [g for g in targets if g in genes]
print("Found:", found)

Found: ['Cdkn1a', 'Myc', 'E2f1', 'Ccnd1', 'Cdk2']


In [9]:
import scanpy as sc
import numpy as np
import pandas as pd

# Load the subsampled file
adata = sc.read_h5ad("multiperturb_seq_subsampled_with_pt.h5ad")

# Set the gene names as the index (from the 'gene' column)
# Convert to plain list to avoid categorical issues
adata.var_names = adata.var['gene'].values.astype(str)

# Make unique if needed (some symbols may be duplicated)
if not adata.var_names.is_unique:
    print("Making gene names unique...")
    adata.var_names_make_unique()

# Ensure ATAC_gene key exists
if 'ATAC_gene' not in adata.obsm and 'ATAC' in adata.obsm:
    adata.obsm['ATAC_gene'] = adata.obsm['ATAC']

# Verify
print("First 10 gene names:", adata.var_names[:10])
print("Has pseudotime?", 'pseudotime' in adata.obs)
print("Has guide_target?", 'guide_target' in adata.obs)  # rename if needed
if 'guide_gene' in adata.obs and 'guide_target' not in adata.obs:
    adata.obs.rename(columns={'guide_gene': 'guide_target'}, inplace=True)

# Save the fixed file
adata.write("multiperturb_seq_ready_for_validation.h5ad")

Making gene names unique...
First 10 gene names: Index(['TSPAN6', 'TNMD', 'DPM1', 'SCYL3', 'C1orf112', 'FGR', 'CFH', 'FUCA2',
       'GCLC', 'STPG1'],
      dtype='object')
Has pseudotime? True
Has guide_target? True


In [1]:
import scanpy as sc
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu

# Load your full AnnData (the one with gene symbols as var_names)
adata = sc.read_h5ad("multiperturb_seq_ready_for_validation.h5ad")

# Keep the subset columns
print(adata.obs.columns)

# Extract control and perturbed cell indices
control_label = 'non-targeting (mouse)'
perturb_label = 'SMARCE1'

is_control = adata.obs['guide_target'] == control_label
is_perturbed = adata.obs['guide_target'] == perturb_label

# Get the raw counts (assuming adata.X is log1p transformed; use .layers['counts'] if available)
# If you have raw counts in adata.layers['counts'], use that; otherwise, we'll work with log1p for fold change.
X = adata.X
if hasattr(X, 'toarray'):
    X = X.toarray()

# Compute mean expression in control and perturbed
ctrl_mean = X[is_control].mean(axis=0)
pert_mean = X[is_perturbed].mean(axis=0)

# Fold change (log scale, pseudo‑count to avoid division by zero)
fold_change = pert_mean - ctrl_mean   # because already log1p

# Rank genes by absolute fold change
abs_fc = np.abs(fold_change)
top_idx = np.argsort(-abs_fc)[:100]   # top 100 genes by fold change

# Get gene symbols for those top genes
target_genes_candidates = adata.var_names[top_idx].tolist()
print("Top 10 candidate target genes:", target_genes_candidates[:10])

Index(['guide_id', 'guide_target', 'pseudotime'], dtype='object')
Top 10 candidate target genes: ['LINC00486', 'MALAT1', '-3285', 'Malat1', '-3340', 'TCF12', '-3350', 'ANKRD11', 'TMTC2', 'ENSMUSG00000098178']
